# CMIP variable prediction — EXP4

To run this notebook with a job :

qsub -N exp4_run1 -v RUN_TAG=run1,NUM_SAMPLE=500000,AUTOENCODER_TYPE=CNN,ALIGNMENT_METHOD=swd,VARIABLE=pr,VAL_FRACTION=0.05,TEST_FRACTION=0.15,LAMBDA_ALIGN=0.0001 run_exp4_pipeline.pbs

### Data preprocessing

**Library import**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import torch
import torch.nn as nn
import json
from pathlib import Path

EXPERIENCE 4 HYPERPARAMETERS:

In [ ]:
setup_name = "exp4"

In [ ]:
num_sample = 1000000
chosen_autoencoder_type = "CNN" # choose between "MLP" and "CNN"
inv_alignment_method = "swd"  # "swd" for sliced Wasserstein distance, and "adversarial" for adversarial-classifier
variable = "pr"
val_fraction = 0.05
test_fraction = 0.15
cera_lambda_align = 0.0001

In [ ]:
# ========================================
# Autoencoder Hyperparameters
# ========================================
ae_latent_dim = 64

# ========================================
# Invariant AE Hyperparameters (Exp4)
# ========================================

inv_n_epochs = 100

inv_align_dims = 48
inv_batch_size = 256
inv_patience = 8
inv_learning_rate = 1e-3
inv_weight_decay = 1e-3
inv_n_projections = 64

**Data Loading**

You have to run a PBS job to create the data loaded in the next cell. In the pbs file you can choose the following parameters :

- --max-abs-lat value \ (recommanded : 30)
- --patch-size-km value \ (recommanded : 1000)
- --time-stride value \ (recommanded : 24)
- --max-samples-per-climate value \ (recommanded : 1000)
- --random-seed value \ (recommanded : 42)

to run the PBS job use the following command in /glade/u/home/tsalin/CMIP: 

qsub run_build_multivariate_samples.pbs

to follow what's going on : 

qstat -u tsalin

In [ ]:
precomputed_dir = Path(f"/glade/derecho/scratch/tsalin/CMIP/derived/multivariate_samples_optimized_NGS_v{num_sample}")
if not precomputed_dir.exists():
    raise FileNotFoundError(f"Precomputed data directory not found: {precomputed_dir}")

with open(precomputed_dir / "run_config.json", "r", encoding="utf-8") as f:
    run_cfg = json.load(f)

climate_order = list(run_cfg["climate_order"])
climate_colors = dict(run_cfg["climate_colors"])
selected_variables_full = list(run_cfg["selected_variables"])
max_abs_lat = float(run_cfg["max_abs_lat"])
patch_size_km = float(run_cfg["patch_size_km"])
time_stride = int(run_cfg["time_stride"])
max_samples_per_climate = int(run_cfg["max_samples_per_climate"])
random_seed = int(run_cfg["random_seed"])
n_lat = int(run_cfg["n_lat"])
n_lon = int(run_cfg["n_lon"])
grid_points_per_patch = int(run_cfg["grid_points_per_patch"])
n_patches = int(run_cfg["n_patches"])

features_by_climate_full = {
    c: np.load(precomputed_dir / f"features_{c}.npy")
    for c in climate_order
}
metadata_by_climate = {
    c: pd.read_csv(precomputed_dir / f"metadata_{c}.csv")
    for c in climate_order
}

# Split off variable as a dedicated label while keeping sample/grid-point alignment.
if variable not in selected_variables_full:
    raise ValueError(f"Variable '{variable}' was not found in run_config selected_variables.")

n_variables_full = len(selected_variables_full)
expected_dim_full = n_variables_full * grid_points_per_patch
variable_index = selected_variables_full.index(variable)
variable_col_start = variable_index * grid_points_per_patch
variable_col_end = variable_col_start + grid_points_per_patch

feature_mask_without_variable = np.ones(expected_dim_full, dtype=bool)
feature_mask_without_variable[variable_col_start:variable_col_end] = False

label_variable_by_climate = {}
features_by_climate = {}
for c in climate_order:
    X_full = features_by_climate_full[c]
    if X_full.shape[1] != expected_dim_full:
        raise ValueError(
            f"Unexpected feature dimension for {c}: got {X_full.shape[1]}, expected {expected_dim_full}."
        )

    # Keep variable values (one value per patch grid point) as label for each sample.
    label_variable_by_climate[c] = X_full[:, variable_col_start:variable_col_end].copy()

    # Keep all non-variable variables as model features used in the rest of the notebook.
    features_by_climate[c] = X_full[:, feature_mask_without_variable]

selected_variables = [v for v in selected_variables_full if v != variable]

# Convenience aggregate preserving same sample order as stacked climate features.
label_variable = np.vstack([label_variable_by_climate[c] for c in climate_order])

sample_count_df = pd.read_csv(precomputed_dir / "sample_count.csv").set_index("scenario")
pre_sampling_df = pd.read_csv(precomputed_dir / "pre_sampling_df.csv").set_index("scenario")
patch_catalog = pd.read_csv(precomputed_dir / "patch_catalog.csv")
sampling_diagnostics_df = pd.read_csv(precomputed_dir / "sampling_diagnostics.csv").set_index("scenario")
nan_summary_by_variable_df = pd.read_csv(precomputed_dir / "nan_summary_by_variable.csv")
sample_pairs_by_climate = {
    c: pd.read_csv(precomputed_dir / f"sample_pairs_{c}.csv")
    for c in climate_order
}

display(sample_count_df)
display(pre_sampling_df)
display(sampling_diagnostics_df)
print(f"Feature variables used downstream (without {variable}): {selected_variables}")
print(f"label_{variable} shape (all samples x grid points): {label_variable.shape}")

In [ ]:
# Backward-compatible aliases used later in the notebook
climate_order = list(climate_order)
climate_colors = dict(climate_colors)
sample_count_df = sample_count_df.copy()
pre_sampling_df = pre_sampling_df.copy()
patch_catalog = patch_catalog.copy()
sampling_diagnostics_df = sampling_diagnostics_df.copy()
nan_summary_by_variable_df = nan_summary_by_variable_df.copy()
inv_train_climates = ["historical", "ssp245"]

In [ ]:
# Adversarial classifier architecture (used only when inv_alignment_method == "adversarial").
inv_classifier_hidden_dim_1 = 128
inv_classifier_hidden_dim_2 = 64

# Checkpoint: path to save/resume training state between PBS jobs.
import re as _re

def _sanitize_ckpt(val):
    return _re.sub(r'[^A-Za-z0-9._-]+', '_', str(val).strip())

_ckpt_suffix = "_".join([
    _sanitize_ckpt(setup_name),
    f"ns{_sanitize_ckpt(num_sample)}",
    _sanitize_ckpt(chosen_autoencoder_type),
    _sanitize_ckpt(inv_alignment_method),
    _sanitize_ckpt(variable),
    _sanitize_ckpt(val_fraction),
    _sanitize_ckpt(test_fraction),
    _sanitize_ckpt(cera_lambda_align),
])
inv_checkpoint_path = Path(
    f"/glade/u/home/tsalin/CMIP/model_evaluation/exp4/inv_checkpoint_{_ckpt_suffix}.pt"
)
inv_checkpoint_freq = 1  # Save checkpoint every N epochs (1 = every epoch)

if ae_latent_dim < inv_align_dims:
    raise ValueError(f"ae_latent_dim={ae_latent_dim} must be >= inv_align_dims={inv_align_dims}")

**Latitude Band, 1000-km Patches, and Multivariate Samples**

This step builds one unified multivariate dataset used by all downstream analyses.

Workflow:
- select data inside +/- max_abs_lat (default: 30 deg);
- split the region into non-overlapping ~1000 km x 1000 km patches;
- for each climate, build samples from all variables and all lat/lon points inside each patch at sampled times;
- aggregate all climates to fit one common scaler

Utilities :

In [ ]:
torch.manual_seed(random_seed)
np.random.seed(random_seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Train/validation/test split, followed by rigorous standardization

In [ ]:
def build_split_indices(data_by_climate, val_fraction=0.10, test_fraction=0.20, seed=42):
    """Build train/val/test split indices separately within each climate."""
    split_indices = {}
    rng = np.random.default_rng(seed)

    for climate, X in data_by_climate.items():
        n = X.shape[0]
        indices = np.arange(n)
        rng.shuffle(indices)

        n_test = max(1, int(round(test_fraction * n)))
        n_val = max(1, int(round(val_fraction * n)))
        n_train = max(1, n - n_val - n_test)

        train_idx = indices[:n_train]
        val_idx = indices[n_train:n_train + n_val]
        test_idx = indices[n_train + n_val:]

        split_indices[climate] = {
            "train": train_idx,
            "val": val_idx,
            "test": test_idx,
        }

    return split_indices


# Keep explicit raw aliases. The model will still be trained on standardized data,
# but normalization statistics are fit below from raw historical-train samples only.
raw_features_by_climate = {
    climate: np.asarray(X, dtype=np.float32)
    for climate, X in features_by_climate.items()
}
raw_label_variable_by_climate = {
    climate: np.asarray(label_variable_by_climate[climate], dtype=np.float32)
    for climate in climate_order
}

# Split raw, unstandardized data first.
ae_split_indices = build_split_indices(
    raw_features_by_climate,
    val_fraction=val_fraction,
    test_fraction=test_fraction,
    seed=random_seed,
)

# ── RAM optimization: truncate eval-only climates to their test split ───────
_eval_only_climates = [c for c in climate_order if c not in inv_train_climates]
if _eval_only_climates:
    for c in _eval_only_climates:
        _idx = ae_split_indices[c]["test"]
        _X_sub = raw_features_by_climate[c][_idx]
        raw_features_by_climate[c] = _X_sub
        features_by_climate[c]     = _X_sub
        _y_sub = raw_label_variable_by_climate[c][_idx]
        raw_label_variable_by_climate[c] = _y_sub
        label_variable_by_climate[c]     = _y_sub
        features_by_climate_full[c] = features_by_climate_full[c][_idx]
        metadata_by_climate[c] = metadata_by_climate[c].iloc[_idx].reset_index(drop=True)
        ae_split_indices[c] = {
            "train": np.array([], dtype=np.intp),
            "val":   np.array([], dtype=np.intp),
            "test":  np.arange(len(_idx), dtype=np.intp),
        }
    print(f"RAM opt: {_eval_only_climates} truncated to {len(_idx)} samples (test split).")
    del _idx, _X_sub, _y_sub
del _eval_only_climates
# ── End RAM optimization ──────────────────────────────────────────────────────

# -----------------------------------------------------------------------------
# Rigorous variable-wise standardization
# -----------------------------------------------------------------------------
# Inputs: one mean/std per physical input variable, using all grid points from
# historical train samples only.
# Label: one mean/std for the target variable, using all grid points from
# historical train samples only.
# The fitted statistics are then applied to every climate/split without refit.

reference_climate_for_scaling = "historical"
if reference_climate_for_scaling not in raw_features_by_climate:
    raise ValueError(f"Reference climate {reference_climate_for_scaling!r} not found.")

hist_train_idx = ae_split_indices[reference_climate_for_scaling]["train"]

X_hist_train = np.asarray(
    raw_features_by_climate[reference_climate_for_scaling][hist_train_idx],
    dtype=np.float64,
)
y_hist_train = np.asarray(
    raw_label_variable_by_climate[reference_climate_for_scaling][hist_train_idx],
    dtype=np.float64,
)

n_input_variables = len(selected_variables)
expected_input_dim = n_input_variables * grid_points_per_patch

if X_hist_train.shape[1] != expected_input_dim:
    raise ValueError(
        f"Unexpected input feature dimension: got {X_hist_train.shape[1]}, "
        f"expected {expected_input_dim} = {n_input_variables} variables × {grid_points_per_patch} points."
    )
if y_hist_train.shape[1] != grid_points_per_patch:
    raise ValueError(
        f"Unexpected label dimension: got {y_hist_train.shape[1]}, expected {grid_points_per_patch}."
    )

input_variable_means = np.zeros(n_input_variables, dtype=np.float64)
input_variable_stds = np.ones(n_input_variables, dtype=np.float64)

for var_idx, var_name in enumerate(selected_variables):
    cols_for_var = slice(
        var_idx * grid_points_per_patch,
        (var_idx + 1) * grid_points_per_patch,
    )
    values = X_hist_train[:, cols_for_var].reshape(-1)
    finite_values = values[np.isfinite(values)]

    if finite_values.size == 0:
        raise ValueError(f"No finite historical train values found for input variable {var_name!r}.")

    if var_name == "pr":
        finite_values = np.log1p(finite_values * 86400)

    mu = float(np.mean(finite_values))
    sigma = float(np.std(finite_values))
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = 1.0

    input_variable_means[var_idx] = mu
    input_variable_stds[var_idx] = sigma

label_values = y_hist_train.reshape(-1)
label_finite_values = label_values[np.isfinite(label_values)]
if label_finite_values.size == 0:
    raise ValueError(f"No finite historical train label values found for {variable!r}.")

if variable == "pr":
    label_finite_values = np.log1p(label_finite_values * 86400)

label_variable_mean = float(np.mean(label_finite_values))
label_variable_std = float(np.std(label_finite_values))
if not np.isfinite(label_variable_std) or label_variable_std <= 0:
    label_variable_std = 1.0


def standardize_input_variablewise(X_raw):
    """Apply historical-train variable-wise normalization to the input variables."""
    X_raw = np.asarray(X_raw, dtype=np.float64)
    X_scaled = np.empty_like(X_raw, dtype=np.float32)

    for var_idx, var_name in enumerate(selected_variables):
        cols_for_var = slice(
            var_idx * grid_points_per_patch,
            (var_idx + 1) * grid_points_per_patch,
        )
        values = X_raw[:, cols_for_var]
        if var_name == "pr":
            values = np.log1p(values * 86400)
        X_scaled[:, cols_for_var] = (
            (values - input_variable_means[var_idx])
            / input_variable_stds[var_idx]
        ).astype(np.float32)

    return X_scaled


def standardize_label_variablewise(y_raw):
    """Apply historical-train target-variable normalization to all grid points of the label."""
    y_raw = np.asarray(y_raw, dtype=np.float64)
    if variable == "pr":
        y_raw = np.log1p(y_raw * 86400)
    return ((y_raw - label_variable_mean) / label_variable_std).astype(np.float32)


def denormalize_label_variable(y_scaled):
    """Convert standardized target-variable arrays back to physical units (mm/day for pr)."""
    y_scaled = np.asarray(y_scaled, dtype=np.float64)
    result = y_scaled * label_variable_std + label_variable_mean
    if variable == "pr":
        result = np.expm1(result)
    return result.astype(np.float32)

class VariableWiseInputScaler:
    """Scaler-like helper implementing variable-wise normalization/inverse transform."""
    def __init__(self, means, stds, grid_points_per_patch):
        self.means_ = np.asarray(means, dtype=np.float64)
        self.scale_ = np.asarray(stds, dtype=np.float64)
        self.grid_points_per_patch = int(grid_points_per_patch)

    def transform(self, X):
        X = np.asarray(X, dtype=np.float64)
        out = np.empty_like(X, dtype=np.float32)
        for var_idx, var_name in enumerate(selected_variables):
            cols = slice(
                var_idx * self.grid_points_per_patch,
                (var_idx + 1) * self.grid_points_per_patch,
            )
            values = X[:, cols]
            if var_name == "pr":
                values = np.log1p(values * 86400)
            out[:, cols] = ((values - self.means_[var_idx]) / self.scale_[var_idx]).astype(np.float32)
        return out

    def inverse_transform(self, X):
        X = np.asarray(X, dtype=np.float64)
        out = np.empty_like(X, dtype=np.float32)
        for var_idx, var_name in enumerate(selected_variables):
            cols = slice(
                var_idx * self.grid_points_per_patch,
                (var_idx + 1) * self.grid_points_per_patch,
            )
            values = X[:, cols] * self.scale_[var_idx] + self.means_[var_idx]
            if var_name == "pr":
                values = np.expm1(values)
            out[:, cols] = values.astype(np.float32)
        return out


class SingleVariableLabelScaler:
    """Scaler-like helper for the target variable."""
    def __init__(self, mean, std):
        self.mean_ = np.array([float(mean)], dtype=np.float64)
        self.scale_ = np.array([float(std)], dtype=np.float64)

    def transform(self, y):
        y = np.asarray(y, dtype=np.float64)
        if variable == "pr":
            y = np.log1p(y * 86400)
        return ((y - self.mean_[0]) / self.scale_[0]).astype(np.float32)

    def inverse_transform(self, y):
        y = np.asarray(y, dtype=np.float64)
        result = y * self.scale_[0] + self.mean_[0]
        if variable == "pr":
            result = np.expm1(result)
        return result.astype(np.float32)


input_scaler = VariableWiseInputScaler(input_variable_means, input_variable_stds, grid_points_per_patch)
label_scaler = SingleVariableLabelScaler(label_variable_mean, label_variable_std)


scaled_features_by_climate = {
    climate: standardize_input_variablewise(raw_features_by_climate[climate])
    for climate in climate_order
}

scaled_label_variable_by_climate = {
    climate: standardize_label_variablewise(raw_label_variable_by_climate[climate])
    for climate in climate_order
}

# The rest of the notebook trains/evaluates on standardized inputs.
label_variable_by_climate = scaled_label_variable_by_climate

standardization_summary_df = pd.DataFrame({
    "variable": selected_variables + [variable],
    "role": ["input"] * len(selected_variables) + ["label"],
    "mean_fit_on_historical_train_all_points": list(input_variable_means) + [label_variable_mean],
    "std_fit_on_historical_train_all_points": list(input_variable_stds) + [label_variable_std],
})

print("✓ Train/val/test split created before standardization.")
print("✓ Input normalization is variable-wise: one mean/std per input variable using all grid points.")
print("✓ Label normalization is variable-wise: one mean/std for the target variable using all grid points.")
print("✓ All normalization statistics are fitted only on historical train samples.")
print("✓ The same fitted statistics are applied to all climates without refit.")
print("Standardization reference climate:", reference_climate_for_scaling)
print("Historical train samples used for scaling:", len(hist_train_idx))
print("Scaled input shapes:", {c: scaled_features_by_climate[c].shape for c in climate_order})
print("Scaled label shapes:", {c: scaled_label_variable_by_climate[c].shape for c in climate_order})
display(standardization_summary_df)


In [ ]:
# RAM Reduction
del features_by_climate_full
del features_by_climate, raw_features_by_climate
del raw_label_variable_by_climate

Different structures for the AE :

Structure inspired by CERA (CNN2D) - it uses the spatial structure of the samples

In [ ]:
class CNNEncoder(nn.Module):
    def __init__(self, input_channels, spatial_shape, latent_dim):
        super().__init__()
        self.input_channels = input_channels
        self.spatial_shape = spatial_shape
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, input_channels, *spatial_shape)
            feature_map = self.features(dummy)
            self.feature_shape = tuple(feature_map.shape[1:])
            self.flatten_dim = int(np.prod(self.feature_shape))
        self.projection = nn.Linear(self.flatten_dim, latent_dim)

    def forward(self, x):
        if x.ndim == 2:
            x = x.reshape(x.shape[0], self.input_channels, *self.spatial_shape)
        elif x.ndim != 4:
            raise ValueError("Expected a 2D flat batch or a 4D image batch.")
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        return self.projection(x)


class CNNDecoder(nn.Module):
    def __init__(self, output_channels, spatial_shape, latent_dim, feature_shape):
        super().__init__()
        self.output_channels = output_channels
        self.spatial_shape = spatial_shape
        self.feature_shape = feature_shape
        self.project = nn.Sequential(
            nn.Linear(latent_dim, int(np.prod(feature_shape))),
            nn.ReLU(inplace=True),
        )
        self.refine = nn.Sequential(
            nn.Conv2d(feature_shape[0], 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(size=spatial_shape, mode="bilinear", align_corners=False),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, output_channels, kernel_size=3, padding=1),
        )

    def forward(self, z):
        x = self.project(z)
        x = x.reshape(z.shape[0], *self.feature_shape)
        x = self.refine(x)
        return torch.flatten(x, start_dim=1)


class CNNAutoEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dims=None):
        super().__init__()
        _ = hidden_dims
        self.input_channels = len(selected_variables)
        self.spatial_shape = (n_lat, n_lon)

        expected_dim = self.input_channels * grid_points_per_patch
        if input_dim != expected_dim:
            raise ValueError(
                f"input_dim={input_dim} is incompatible with a CNN reshape using "
                f"{self.input_channels} channels and {grid_points_per_patch} grid points per patch."
            )

        self.encoder = CNNEncoder(self.input_channels, self.spatial_shape, latent_dim)
        self.decoder = CNNDecoder(self.input_channels, self.spatial_shape, latent_dim, self.encoder.feature_shape)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

A more classical structure using only MLPs - it doesn't use the spatial structure but apparently it performs better

In [ ]:
input_dim = next(iter(scaled_features_by_climate.values())).shape[1]
MLP_hidden_dim_1 = max(256, min(1024, input_dim // 2))
MLP_hidden_dim_2 = max(128, min(512, input_dim // 8))


class MLPEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dims=(512, 256)):
        super().__init__()
        h1, h2 = hidden_dims
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Linear(h2, latent_dim),
        )

    def forward(self, x):
        return self.net(x)


class MLPDecoder(nn.Module):
    def __init__(self, latent_dim, output_dim, hidden_dims=(256, 512)):
        super().__init__()
        h1, h2 = hidden_dims
        self.net = nn.Sequential(
            nn.Linear(latent_dim, h1),
            nn.ReLU(),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Linear(h2, output_dim),
        )

    def forward(self, z):
        return self.net(z)


class MLPAutoEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dims=(512, 256)):
        super().__init__()
        self.encoder = MLPEncoder(input_dim, latent_dim, hidden_dims=hidden_dims)
        self.decoder = MLPDecoder(latent_dim, input_dim, hidden_dims=hidden_dims[::-1])

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z


# Fourth Experiment - Invariant Autoencoder with Latent Alignment

In this fourth experiment, we dive a step closer to the real CERA architecture by adding an explicit climate invariance term to the autoencoder loss.

The idea is to test wether adding an alignment loss between climates will effectivelly bring different climate distributions closer compared to the raw data and to the simple AE architecture. Here we only train the AE on the historical climate and on SSP245 to reproduce the CERA architecture (one "cold" and one "warm" climate).

Training set:
- **historical + ssp245**

Loss:
- reconstruction loss on all samples;
- alignment loss between latent samples from **historical** and **ssp245**.

Loss equation : 
$$
L = L_{\text{rec}} + \lambda_{\text{Align}} \cdot \text{Align}(Z^{\text{hist}}_{\text{align}}, Z^{\text{ssp245}}_{\text{align}})
$$

Alignment method : 

Here we consider two different method to align the historical and SSP245 climates ;
- We consider a sliced Wasserstein alignment loss 
- And a adversarial classifier.

Note that in CERA, the method used is Earth Mover's Distance (EMD), but EMD can be expensive in high dimension, it is why we use a sliced Wasserstein alignment loss, which is a practical EMD-style approximation.

Note that :
- reconstruction loss on all 64 latent dimensions;
- alignment applied only to the first 48 latent dimensions.

Note that the AE is either using a CNN or a simple MLP based on the setting of the Third Experiment

General architecture of the AE :

In [ ]:
def _build_autoencoder_by_type(autoencoder_type: str, input_dim: int, latent_dim: int):
    if autoencoder_type == "CNN":
        return CNNAutoEncoder(
            input_dim=input_dim,
            latent_dim=latent_dim,
        )
    if autoencoder_type == "MLP":
        return MLPAutoEncoder(
            input_dim=input_dim,
            latent_dim=latent_dim,
            hidden_dims=(MLP_hidden_dim_1, MLP_hidden_dim_2),
        )
    raise ValueError(f"Unsupported autoencoder type: {autoencoder_type}")

Architecture and training function of the SWD loss :

In [ ]:
def sliced_wasserstein_alignment_loss_torch(z_hist: torch.Tensor, z_ssp: torch.Tensor, n_projections: int = 64) -> torch.Tensor:
    # z_hist/z_ssp: [batch, align_dims]
    d = z_hist.shape[1]
    dirs = torch.randn(n_projections, d, device=z_hist.device, dtype=z_hist.dtype)
    dirs = dirs / (torch.norm(dirs, dim=1, keepdim=True) + 1e-12)

    proj_hist = z_hist @ dirs.T
    proj_ssp = z_ssp @ dirs.T

    proj_hist_sorted, _ = torch.sort(proj_hist, dim=0)
    proj_ssp_sorted, _ = torch.sort(proj_ssp, dim=0)

    m = min(proj_hist_sorted.shape[0], proj_ssp_sorted.shape[0])
    if m == 0:
        return torch.tensor(0.0, device=z_hist.device, dtype=z_hist.dtype)

    return torch.mean(torch.abs(proj_hist_sorted[:m] - proj_ssp_sorted[:m]))


def _make_loader_for_split(data_by_climate, split_indices, climate, split, batch_size, shuffle):
    idx = split_indices[climate][split]
    X = np.asarray(data_by_climate[climate][idx], dtype=np.float32)
    ds = torch.utils.data.TensorDataset(torch.tensor(X, dtype=torch.float32))
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=True)


def train_invariant_autoencoder(
    train_climates,
    latent_dim=64,
    n_epochs=30,
    lr=1e-3,
    batch_size=256,
    weight_decay=1e-5,
    align_dims=48,
    alignment_weight=0.2,
    n_projections=64,
    checkpoint_path=None,
    checkpoint_freq=1,
):
    if train_climates != ["historical", "ssp245"]:
        raise ValueError("This invariant setup expects train_climates=['historical', 'ssp245']")

    input_dim_local = next(iter(scaled_features_by_climate.values())).shape[1]
    model = _build_autoencoder_by_type(chosen_autoencoder_type, input_dim_local, latent_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Use existing Third-Experiment splits for strict comparability.
    train_loader_hist = _make_loader_for_split(scaled_features_by_climate, ae_split_indices, "historical", "train", batch_size // 2, True)
    train_loader_ssp = _make_loader_for_split(scaled_features_by_climate, ae_split_indices, "ssp245", "train", batch_size // 2, True)
    val_loader_hist = _make_loader_for_split(scaled_features_by_climate, ae_split_indices, "historical", "val", batch_size // 2, False)
    val_loader_ssp = _make_loader_for_split(scaled_features_by_climate, ae_split_indices, "ssp245", "val", batch_size // 2, False)

    best_state = None
    best_val = np.inf
    best_epoch = -1
    patience_counter = 0
    history = []

    start_epoch = 1
    if checkpoint_path is not None:
        _ckpt_path = Path(checkpoint_path)
        if _ckpt_path.exists():
            _ckpt = torch.load(_ckpt_path, map_location=device)
            model.load_state_dict(_ckpt["model_state"])
            optimizer.load_state_dict(_ckpt["optimizer_state"])
            start_epoch      = _ckpt["epoch"] + 1
            best_val         = _ckpt["best_val"]
            best_epoch       = _ckpt["best_epoch"]
            patience_counter = _ckpt["patience_counter"]
            best_state       = _ckpt["best_state"]
            history          = _ckpt["history"]
            print(f"[CHECKPOINT] Resuming from epoch {start_epoch} "
                  f"(best: epoch {best_epoch}, val={best_val:.6f})")

    for epoch in range(start_epoch, n_epochs + 1):
        model.train()
        train_recon_losses = []
        train_align_losses = []

        n_steps = min(len(train_loader_hist), len(train_loader_ssp))
        hist_iter = iter(train_loader_hist)
        ssp_iter = iter(train_loader_ssp)

        for _ in range(n_steps):
            xb_hist = next(hist_iter)[0].to(device)
            xb_ssp = next(ssp_iter)[0].to(device)
            xb = torch.cat([xb_hist, xb_ssp], dim=0)

            optimizer.zero_grad()
            x_hat, z = model(xb)
            recon_loss = torch.mean((x_hat - xb) ** 2)

            z_hist = z[: xb_hist.shape[0], :align_dims]
            z_ssp = z[xb_hist.shape[0] :, :align_dims]
            align_loss = sliced_wasserstein_alignment_loss_torch(z_hist, z_ssp, n_projections=n_projections)

            total_loss = (1 - alignment_weight) * recon_loss + alignment_weight * align_loss
            total_loss.backward()
            optimizer.step()

            train_recon_losses.append(float(recon_loss.detach().cpu().item()))
            train_align_losses.append(float(align_loss.detach().cpu().item()))

        model.eval()
        val_recon_losses = []
        val_align_losses = []
        with torch.no_grad():
            n_val_steps = min(len(val_loader_hist), len(val_loader_ssp))
            hist_val_iter = iter(val_loader_hist)
            ssp_val_iter = iter(val_loader_ssp)

            for _ in range(n_val_steps):
                xb_hist = next(hist_val_iter)[0].to(device)
                xb_ssp = next(ssp_val_iter)[0].to(device)
                xb = torch.cat([xb_hist, xb_ssp], dim=0)

                x_hat, z = model(xb)
                recon_loss = torch.mean((x_hat - xb) ** 2)
                z_hist = z[: xb_hist.shape[0], :align_dims]
                z_ssp = z[xb_hist.shape[0] :, :align_dims]
                align_loss = sliced_wasserstein_alignment_loss_torch(z_hist, z_ssp, n_projections=n_projections)

                val_recon_losses.append(float(recon_loss.detach().cpu().item()))
                val_align_losses.append(float(align_loss.detach().cpu().item()))

        train_recon = float(np.mean(train_recon_losses)) if train_recon_losses else np.nan
        train_align = float(np.mean(train_align_losses)) if train_align_losses else np.nan
        val_recon = float(np.mean(val_recon_losses)) if val_recon_losses else np.nan
        val_align = float(np.mean(val_align_losses)) if val_align_losses else np.nan
        val_total = (1 - alignment_weight) * val_recon + alignment_weight * val_align

        history.append(
            {
                "epoch": epoch,
                "train_recon_loss": train_recon,
                "train_align_loss": train_align,
                "train_total_loss": (1 - alignment_weight) * train_recon + alignment_weight * train_align,
                "val_recon_loss": val_recon,
                "val_align_loss": val_align,
                "val_total_loss": val_total,
            }
        )

        if checkpoint_path is not None and epoch % checkpoint_freq == 0:
            torch.save({
                "epoch": epoch,
                "model_state": {k: v.cpu().clone() for k, v in model.state_dict().items()},
                "optimizer_state": optimizer.state_dict(),
                "best_val": best_val,
                "best_epoch": best_epoch,
                "patience_counter": patience_counter,
                "best_state": best_state,
                "history": history,
            }, checkpoint_path)

        if val_total < best_val:
            best_val = val_total
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= inv_patience:
                if checkpoint_path is not None:
                    torch.save({
                        "epoch": epoch,
                        "model_state": {k: v.cpu().clone() for k, v in model.state_dict().items()},
                        "optimizer_state": optimizer.state_dict(),
                        "best_val": best_val,
                        "best_epoch": best_epoch,
                        "patience_counter": patience_counter,
                        "best_state": best_state,
                        "history": history,
                    }, checkpoint_path)
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history), best_epoch

Architecture and training function of the adversarial classifier

In [ ]:
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None

def gradient_reversal(x: torch.Tensor, lambd: float = 1.0) -> torch.Tensor:
    return GradientReversalFunction.apply(x, lambd)

class LatentDomainClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dims=(128, 64), n_domains: int = 2):
        super().__init__()
        hidden_dim_1, hidden_dim_2 = hidden_dims
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim_1),
            nn.ReLU(),
            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.ReLU(),
            nn.Linear(hidden_dim_2, n_domains),
        )

    def forward(self, z: torch.Tensor, grl_lambda: float = 1.0) -> torch.Tensor:
        return self.net(gradient_reversal(z, grl_lambda))

def train_invariant_autoencoder_adversarial(
    train_climates,
    latent_dim=64,
    n_epochs=30,
    lr=1e-3,
    batch_size=256,
    weight_decay=1e-5,
    align_dims=48,
    alignment_weight=0.2,
    classifier_hidden_dims=(128, 64),
    checkpoint_path=None,
    checkpoint_freq=1,
):
    input_dim_local = next(iter(scaled_features_by_climate.values())).shape[1]
    model = _build_autoencoder_by_type(chosen_autoencoder_type, input_dim_local, latent_dim).to(device)
    domain_classifier = LatentDomainClassifier(align_dims, hidden_dims=classifier_hidden_dims).to(device)
    optimizer = torch.optim.AdamW(list(model.parameters()) + list(domain_classifier.parameters()), lr=lr, weight_decay=weight_decay)
    ce_loss_fn = nn.CrossEntropyLoss()

    train_loader_hist = _make_loader_for_split(scaled_features_by_climate, ae_split_indices, "historical", "train", batch_size // 2, True)
    train_loader_ssp = _make_loader_for_split(scaled_features_by_climate, ae_split_indices, "ssp245", "train", batch_size // 2, True)
    val_loader_hist = _make_loader_for_split(scaled_features_by_climate, ae_split_indices, "historical", "val", batch_size // 2, False)
    val_loader_ssp = _make_loader_for_split(scaled_features_by_climate, ae_split_indices, "ssp245", "val", batch_size // 2, False)

    best_state = None
    best_classifier_state = None
    best_val = np.inf
    best_epoch = -1
    patience_counter = 0
    history = []

    start_epoch = 1
    if checkpoint_path is not None:
        _ckpt_path = Path(checkpoint_path)
        if _ckpt_path.exists():
            _ckpt = torch.load(_ckpt_path, map_location=device)
            model.load_state_dict(_ckpt["model_state"])
            domain_classifier.load_state_dict(_ckpt["domain_classifier_state"])
            optimizer.load_state_dict(_ckpt["optimizer_state"])
            start_epoch           = _ckpt["epoch"] + 1
            best_val              = _ckpt["best_val"]
            best_epoch            = _ckpt["best_epoch"]
            patience_counter      = _ckpt["patience_counter"]
            best_state            = _ckpt["best_state"]
            best_classifier_state = _ckpt["best_classifier_state"]
            history               = _ckpt["history"]
            print(f"[CHECKPOINT] Resuming from epoch {start_epoch} "
                  f"(best: epoch {best_epoch}, val={best_val:.6f})")

    for epoch in range(start_epoch, n_epochs + 1):
        model.train()
        domain_classifier.train()
        train_recon_losses = []
        train_align_losses = []

        n_steps = min(len(train_loader_hist), len(train_loader_ssp))
        hist_iter = iter(train_loader_hist)
        ssp_iter = iter(train_loader_ssp)

        for _ in range(n_steps):
            xb_hist = next(hist_iter)[0].to(device)
            xb_ssp = next(ssp_iter)[0].to(device)
            xb = torch.cat([xb_hist, xb_ssp], dim=0)

            optimizer.zero_grad()
            x_hat, z = model(xb)
            recon_loss = torch.mean((x_hat - xb) ** 2)
            domain_targets = torch.cat([
                torch.zeros(xb_hist.shape[0], dtype=torch.long, device=device),
                torch.ones(xb_ssp.shape[0], dtype=torch.long, device=device),
            ], dim=0)
            domain_logits = domain_classifier(z[:, :align_dims], grl_lambda=1.0)
            align_loss = ce_loss_fn(domain_logits, domain_targets)
            total_loss = (1 - alignment_weight) * recon_loss + alignment_weight * align_loss
            total_loss.backward()
            optimizer.step()

            train_recon_losses.append(float(recon_loss.detach().cpu().item()))
            train_align_losses.append(float(align_loss.detach().cpu().item()))

        model.eval()
        domain_classifier.eval()
        val_recon_losses = []
        val_align_losses = []
        with torch.no_grad():
            n_val_steps = min(len(val_loader_hist), len(val_loader_ssp))
            hist_val_iter = iter(val_loader_hist)
            ssp_val_iter = iter(val_loader_ssp)

            for _ in range(n_val_steps):
                xb_hist = next(hist_val_iter)[0].to(device)
                xb_ssp = next(ssp_val_iter)[0].to(device)
                xb = torch.cat([xb_hist, xb_ssp], dim=0)

                x_hat, z = model(xb)
                recon_loss = torch.mean((x_hat - xb) ** 2)
                domain_targets = torch.cat([
                    torch.zeros(xb_hist.shape[0], dtype=torch.long, device=device),
                    torch.ones(xb_ssp.shape[0], dtype=torch.long, device=device),
                ], dim=0)
                domain_logits = domain_classifier(z[:, :align_dims], grl_lambda=1.0)
                align_loss = ce_loss_fn(domain_logits, domain_targets)

                val_recon_losses.append(float(recon_loss.detach().cpu().item()))
                val_align_losses.append(float(align_loss.detach().cpu().item()))

        train_recon = float(np.mean(train_recon_losses)) if train_recon_losses else np.nan
        train_align = float(np.mean(train_align_losses)) if train_align_losses else np.nan
        val_recon = float(np.mean(val_recon_losses)) if val_recon_losses else np.nan
        val_align = float(np.mean(val_align_losses)) if val_align_losses else np.nan
        val_total = (1 - alignment_weight) * val_recon + alignment_weight * val_align

        history.append(
            {
                "epoch": epoch,
                "train_recon_loss": train_recon,
                "train_align_loss": train_align,
                "train_total_loss": (1 - alignment_weight) * train_recon + alignment_weight * train_align,
                "train_encoder_effective_loss": train_recon - alignment_weight * train_align,
                "val_recon_loss": val_recon,
                "val_align_loss": val_align,
                "val_total_loss": val_total,
                "val_encoder_effective_loss": val_recon - alignment_weight * val_align,
            }
        )

        if checkpoint_path is not None and epoch % checkpoint_freq == 0:
            torch.save({
                "epoch": epoch,
                "model_state": {k: v.cpu().clone() for k, v in model.state_dict().items()},
                "domain_classifier_state": {k: v.cpu().clone() for k, v in domain_classifier.state_dict().items()},
                "optimizer_state": optimizer.state_dict(),
                "best_val": best_val,
                "best_epoch": best_epoch,
                "patience_counter": patience_counter,
                "best_state": best_state,
                "best_classifier_state": best_classifier_state,
                "history": history,
            }, checkpoint_path)

        if val_total < best_val:
            best_val = val_total
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_classifier_state = {k: v.detach().cpu().clone() for k, v in domain_classifier.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= inv_patience:
                if checkpoint_path is not None:
                    torch.save({
                        "epoch": epoch,
                        "model_state": {k: v.cpu().clone() for k, v in model.state_dict().items()},
                        "domain_classifier_state": {k: v.cpu().clone() for k, v in domain_classifier.state_dict().items()},
                        "optimizer_state": optimizer.state_dict(),
                        "best_val": best_val,
                        "best_epoch": best_epoch,
                        "patience_counter": patience_counter,
                        "best_state": best_state,
                        "best_classifier_state": best_classifier_state,
                        "history": history,
                    }, checkpoint_path)
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    if best_classifier_state is not None:
        domain_classifier.load_state_dict(best_classifier_state)

    return model, pd.DataFrame(history), best_epoch, domain_classifier

Training

In [ ]:
if inv_alignment_method == "swd":
    inv_ae_model, inv_ae_history_df, inv_ae_best_epoch = train_invariant_autoencoder(
        train_climates=inv_train_climates,
        latent_dim=ae_latent_dim,
        n_epochs=inv_n_epochs,
        lr=inv_learning_rate,
        batch_size=inv_batch_size,
        weight_decay=inv_weight_decay,
        align_dims=inv_align_dims,
        alignment_weight=cera_lambda_align,
        n_projections=inv_n_projections,
        checkpoint_path=inv_checkpoint_path,
        checkpoint_freq=inv_checkpoint_freq,
    )

elif inv_alignment_method == "adversarial":
    inv_ae_model, inv_ae_history_df, inv_ae_best_epoch, inv_domain_classifier = train_invariant_autoencoder_adversarial(
        train_climates=inv_train_climates,
        latent_dim=ae_latent_dim,
        n_epochs=inv_n_epochs,
        lr=inv_learning_rate,
        batch_size=inv_batch_size,
        weight_decay=inv_weight_decay,
        align_dims=inv_align_dims,
        alignment_weight=cera_lambda_align,
        classifier_hidden_dims=(inv_classifier_hidden_dim_1, inv_classifier_hidden_dim_2),
        checkpoint_path=inv_checkpoint_path,
        checkpoint_freq=inv_checkpoint_freq,
    )

Let's observe the training curves

In [ ]:
print("Chosen autoencoder type:", chosen_autoencoder_type)
print("Invariant alignment method:", inv_alignment_method)
print("Invariant training climates:", ", ".join(inv_train_climates))
print("Best epoch (invariant AE):", inv_ae_best_epoch)

display(inv_ae_history_df.tail())

fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
ax.plot(inv_ae_history_df["epoch"], inv_ae_history_df["train_total_loss"], label="train total")
ax.plot(inv_ae_history_df["epoch"], inv_ae_history_df["val_total_loss"], label="val total")
ax.plot(inv_ae_history_df["epoch"], inv_ae_history_df["train_recon_loss"], "--", label="train recon")
ax.plot(inv_ae_history_df["epoch"], inv_ae_history_df["val_recon_loss"], "--", label="val recon")
ax.plot(inv_ae_history_df["epoch"], inv_ae_history_df["train_align_loss"], ":", label="train align")
ax.plot(inv_ae_history_df["epoch"], inv_ae_history_df["val_align_loss"], ":", label="val align")

if "train_encoder_effective_loss" in inv_ae_history_df.columns:
    ax.plot(inv_ae_history_df["epoch"], inv_ae_history_df["train_encoder_effective_loss"], label="train encoder effective")
if "val_encoder_effective_loss" in inv_ae_history_df.columns:
    ax.plot(inv_ae_history_df["epoch"], inv_ae_history_df["val_encoder_effective_loss"], label="val encoder effective")

ax.set_title("Invariant AE training curves")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.grid(alpha=0.25)
ax.legend(ncol=3)
plt.show()

### Evaluation : Reconstruction quality on test data - EXTRACTION

We evaluate reconstruction skill on test splits for all climates, exactly like in the Third Experiment pipeline.

Evaluation :

In [ ]:
def _build_meta_df(component, climate, metadata, latent_dim, align_dim):
    """Build a lightweight metadata DataFrame without storing numerical arrays per row."""
    df = metadata.copy().reset_index(drop=True)
    df = df.drop(columns=["scenario"], errors="ignore")
    df.insert(0, "experiment",        "CMIP_variable_exp4")
    df.insert(1, "component",         component)
    df.insert(2, "component_order",   0)
    df.insert(3, "scenario",          climate)
    df.insert(4, "scenario_order",    int(climate_order.index(climate)))
    df.insert(5, "sample_idx",        range(len(df)))
    df.insert(6, "latent_dim",        int(latent_dim))
    df.insert(7, "align_dim",         int(align_dim))
    return df


reconstruction_value_names = [
    f"{var}@point_{point_idx}"
    for var in selected_variables
    for point_idx in range(grid_points_per_patch)
]

meta_dfs_recon  = []
truth_arrays_recon = []
pred_arrays_recon  = []

inv_ae_model.eval()
with torch.inference_mode():
    for climate in climate_order:
        idx = ae_split_indices[climate]["test"]
        X_scaled = np.asarray(scaled_features_by_climate[climate][idx], dtype=np.float32)
        metadata = metadata_by_climate[climate].iloc[idx].reset_index(drop=True)

        xb = torch.from_numpy(X_scaled).to(device)
        x_hat, _ = inv_ae_model(xb)
        X_hat_scaled = x_hat.detach().cpu().numpy()

        # Store denormalized reconstruction values for physical interpretation.
        X_true_physical = input_scaler.inverse_transform(X_scaled)
        X_hat_physical = input_scaler.inverse_transform(X_hat_scaled)

        meta_dfs_recon.append(_build_meta_df("reconstruction", climate, metadata, ae_latent_dim, inv_align_dims))
        truth_arrays_recon.append(X_true_physical.astype(np.float32))
        pred_arrays_recon.append(X_hat_physical.astype(np.float32))

inv_reconstruction_payload = {
    "meta_reconstruction":        pd.concat(meta_dfs_recon, ignore_index=True),
    "truth_reconstruction":       np.concatenate(truth_arrays_recon, axis=0),
    "pred_reconstruction":        np.concatenate(pred_arrays_recon,  axis=0),
    "reconstruction_value_names": reconstruction_value_names,
}

# Kept for backward compatibility with cells/scripts that expect this name.
inv_reconstruction_delta_df = pd.DataFrame()

print("Invariant AE reconstruction payload — shapes:")
print(f"  meta_reconstruction  : {inv_reconstruction_payload['meta_reconstruction'].shape}")
print(f"  truth_reconstruction : {inv_reconstruction_payload['truth_reconstruction'].shape}")
print(f"  pred_reconstruction  : {inv_reconstruction_payload['pred_reconstruction'].shape}")
print()
print(inv_reconstruction_payload["meta_reconstruction"].groupby("scenario").size().rename("n_samples_reconstruction"))

### Extraction of latent test representations for all climates.

In [ ]:
inv_latent_by_climate = {}
inv_latent_test_metadata_by_climate = {}
inv_latent_score_frames = []

inv_ae_model.eval()
with torch.no_grad():
    for climate in climate_order:
        idx = ae_split_indices[climate]["test"]
        X = np.asarray(scaled_features_by_climate[climate][idx], dtype=np.float32)
        xb = torch.tensor(X, dtype=torch.float32, device=device)
        _, z = inv_ae_model(xb)
        z_np = z.detach().cpu().numpy()

        inv_latent_by_climate[climate] = z_np
        inv_latent_test_metadata_by_climate[climate] = metadata_by_climate[climate].iloc[idx].reset_index(drop=True)
        inv_latent_score_frames.append(
            pd.DataFrame(z_np, columns=[f"latent_{i+1}" for i in range(z_np.shape[1])]).assign(scenario=climate)
        )

In [ ]:
# Delete the checkpoint once the whole notebook has run successfully
if inv_checkpoint_path.exists():
    inv_checkpoint_path.unlink()
    print(f"[CHECKPOINT] Checkpoint deleted: {inv_checkpoint_path}")
else:
    print("[CHECKPOINT] No checkpoint to delete.")